# 第017单元 分布密度与似然

本Notebook是教学实验。A段只以016为直接先修；B段前必须补010。不预设018期望方差知识。所有数据为原创合成。

完成目标：分别说明生成器、概率质量、密度、似然和后验的固定量与变量；再区分数学零概率、非法参数和数值下溢。使用 Restart Kernel and Run All 重跑，不依赖隐藏变量。

## A1 环境与观测
核心计算只用标准库。先确认本Notebook和experiment.py位于同一目录；读取固定观测1110。y是长度n的平坦序列，p是标量。

In [1]:
from pathlib import Path
import math
import json
from itertools import product
import experiment as ex
base = Path.cwd()
assert (base / 'data' / 'observations.csv').is_file(), '请在单元017目录执行'
y = ex.load_binary_csv(base / 'data' / 'observations.csv')
print('观测:', y, 'n =', len(y), '成功次数 k =', sum(y))
print('p=3/4的质量:', [ex.bernoulli_mass(v, .75) for v in (0, 1)])

观测: [1, 1, 1, 0] n = 4 成功次数 k = 3
p=3/4的质量: [0.25, 0.75]


## A2 有限编号生成器
每次均匀抽0、1、2、3之一，前三个记1，最后一个记0，所以p=3/4。这个有限模型不需要连续均匀分布。固定seed可复现序列，seed不是概率参数。

In [2]:
generated = ex.bernoulli_sample(12, 3, 4, seed=17)
assert generated == ex.bernoulli_sample(12, 3, 4, seed=17)
print('seed=17:', generated)
print('端点:', ex.bernoulli_sample(4, 0, 4), ex.bernoulli_sample(4, 4, 4))

seed=17: [0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1]
端点: [0, 0, 0, 0] [1, 1, 1, 1]


## A3 固定观测计算似然
先手算3/256、1/16、27/256，再查看输出。p=3/4相对于p=1/4的似然比为9。

In [3]:
for p in (0, .25, .5, .75, 1):
    print(p, ex.bernoulli_likelihood(y, p), ex.bernoulli_log_likelihood(y, p))
assert ex.bernoulli_likelihood(y, .75) / ex.bernoulli_likelihood(y, .25) == 9

0 0.0 -inf
0.25 0.01171875 -4.446565155811452
0.5 0.0625 -2.772588722239781
0.75 0.10546875 -2.249340578475233
1 0.0 -inf


## A4 有序序列与计数事件
四个序列是互斥事件，对它们的概率求有限和。改变了观测对象，不能混用绝对概率。

In [4]:
sequences = [s for s in product((0, 1), repeat=4) if sum(s) == 3]
count_probability = sum(ex.bernoulli_likelihood(s, .75) for s in sequences)
print('四个序列:', sequences)
print('成功次数等于3的概率:', count_probability, '= 27/64')
assert count_probability == 27/64

四个序列: [(0, 1, 1, 1), (1, 0, 1, 1), (1, 1, 0, 1), (1, 1, 1, 0)]
成功次数等于3的概率: 0.421875 = 27/64


## A5 端点按照事件支持处理
零似然使用扩展log值负无穷，不用0乘log(0)，也不隐藏裁剪参数。

In [5]:
for sample in ([0,0], [1,1], [1,0]):
    print(sample, {p: ex.bernoulli_log_likelihood(sample, p) for p in (0,1)})
for name, call in [('概率越界', lambda: ex.bernoulli_log_likelihood(y, 1.01)),
                   ('布尔标签', lambda: ex.bernoulli_log_likelihood([True,0], .5)),
                   ('空观测', lambda: ex.bernoulli_log_likelihood([], .5))]:
    try:
        call()
    except (ValueError, TypeError) as error:
        print(name, '被拒绝:', type(error).__name__)
    else:
        raise AssertionError('本应拒绝')

[0, 0] {0: 0.0, 1: -inf}
[1, 1] {0: -inf, 1: 0.0}
[1, 0] {0: -inf, 1: -inf}
概率越界 被拒绝: ValueError
布尔标签 被拒绝: ValueError
空观测 被拒绝: ValueError


## A6 直接乘积下溢
数学上的2的负2000次方严格为正。普通乘积返回0是数值范围限制；逐项log之和仍保留有限结果。

In [6]:
long_y = [1] * 2000
print('直接乘积:', ex.bernoulli_likelihood(long_y, .5))
print('log似然:', ex.bernoulli_log_likelihood(long_y, .5))
assert ex.bernoulli_likelihood(long_y, .5) == 0
assert math.isfinite(ex.bernoulli_log_likelihood(long_y, .5))

直接乘积: 0.0
log似然: -1386.2943611198905


## A7 显式先验与后验
同一数据，在等权先验下后验为0.1/0.9；先验9:1时后验各一半。仅归一化似然已经隐含特定先验。

In [7]:
for prior in ([.5,.5], [.9,.1]):
    posterior = ex.finite_posterior(y, [.25,.75], prior)
    print('先验:', prior, '后验:', posterior)
    assert math.isclose(sum(posterior), 1)
try:
    ex.finite_posterior(y, [0,1], [.5,.5])
except ValueError as error:
    print('候选均不支持观测:', type(error).__name__)

先验: [0.5, 0.5] 后验: [0.10000000000000003, 0.8999999999999999]
先验: [0.9, 0.1] 后验: [0.5000000000000002, 0.4999999999999998]
候选均不支持观测: ValueError


# B段开始前 先补010

下面使用积分、小矩形面积和极限。尚未完成010时，请在此暂停，先完成实验A与练习1至6。μ是位置、σ是正尺度；本段不用018的期望或方差定义。

## B1 密度与事件概率
峰高约3.989每秒可以大于1；区间面积约0.6826895才是无单位概率。有限网格面积和精确参考会有微小差别。

In [8]:
peak = ex.gaussian_density(0, 0, .1)
area = ex.midpoint_integral(lambda x: ex.gaussian_density(x,0,.1), -.1, .1, 2000)
print('峰高:', peak, '区间概率近似:', area)
assert peak > 1 and 0 < area < 1
print('矩形密度总面积:', ex.midpoint_integral(lambda x: 4, 0, .25, 100))

峰高: 3.989422804014326 区间概率近似: 0.682689512301314
矩形密度总面积: 1.0


## B2 积分范围和面板数量是两个控制量
对每个bound比较20/200/2000面板，再比较范围。四面板的[−2,2]近似应为0.9631658450。

In [9]:
print('四面板手算对照:', ex.midpoint_integral(ex.gaussian_density, -2, 2, 4))
for bound in (1,2,4,6):
    results = [(m, ex.midpoint_integral(ex.gaussian_density, -bound, bound, m)) for m in (20,200,2000)]
    print('对称界限:', bound, results)

四面板手算对照: 0.9631658448603826
对称界限: 1 [(20, 0.6828912521252752), (200, 0.6826915085715528), (2000, 0.6826895123013141)]
对称界限: 2 [(20, 0.9548592519460659), (200, 0.9545033354594122), (2000, 0.9544997720976152)]
对称界限: 4 [(20, 0.999943377527326), (200, 0.9999367288491683), (2000, 0.9999366582300907)]
对称界限: 6 [(20, 0.9999999988255692), (200, 0.9999999980377235), (2000, 0.9999999980269342)]


## B3 故意错过窄峰
同一σ=0.01、同一区间[−1,1]，仅改变网格。20个面板得到约0.0000297344；2000个面板约1。

In [10]:
for panels in (20,2000):
    value = ex.midpoint_integral(lambda x: ex.gaussian_density(x,0,.01), -1, 1, panels)
    print('面板:', panels, '面积:', value)
print('x=40密度:', ex.gaussian_density(40), 'log密度:', ex.gaussian_log_density(40))

面板: 20 面积: 2.9734390294686117e-05
面板: 2000 面积: 1.0000000000000009
x=40密度: 0.0 log密度: -800.9189385332047


## B4 连续似然与统一换单位
数据固定为[−0.1,0.1]。σ=.1时，两个位置候选的标准化残差平方和为2和4，log似然差为1。统一改毫秒后共同减去2log(1000)。

In [11]:
sample = [-.1,.1]
a = ex.gaussian_log_likelihood(sample, 0, .1)
b = ex.gaussian_log_likelihood(sample, .1, .1)
converted = ex.gaussian_log_likelihood([-100,100], 0, 100)
print('两个log似然:', a,b,'差:',a-b)
print('改成毫秒:',converted,'共同变化:',converted-a)
assert math.isclose(a-b,1,abs_tol=1e-14)
assert math.isclose(converted-a,-2*math.log(1000),abs_tol=1e-13)

两个log似然: 1.7672931195787456 0.7672931195787456 差: 1.0
改成毫秒: -12.048217438385528 共同变化: -13.815510557964274


## B5 参数错误与数值范围错误
零尺度不在参数域；极端相减溢出是机器范围问题；无法分辨的网格也应拒绝。捕获预期异常不算Notebook执行失败。

In [12]:
for name, call in [('零尺度',lambda: ex.gaussian_density(0,0,0)),
                   ('极端相减',lambda: ex.gaussian_log_density(1e308,-1e308,1)),
                   ('无法分辨网格',lambda: ex.midpoint_integral(lambda x: 1.,1e16,1e16+4,100))]:
    try:
        call()
    except (ValueError, ArithmeticError) as error:
        print(name, '被拒绝:', type(error).__name__)
    else:
        raise AssertionError('本应拒绝')

零尺度 被拒绝: ValueError
极端相减 被拒绝: ArithmeticError
无法分辨网格 被拒绝: ArithmeticError


## 检查 运行独立oracle测试
9组测试包括2142个精确序列、230个有限编号映射、21个对数参考、72个Gaussian点参考、4个级数积分参考与47种拒绝输入。参考算法与被测计算路径不同；测试覆盖不等于全输入证明。

In [13]:
import unittest
import test_experiment
suite = unittest.defaultTestLoader.loadTestsFromModule(test_experiment)
result = unittest.TextTestRunner(verbosity=1).run(suite)
assert result.wasSuccessful()

.

.

.

.

.

.

.

.

.

----------------------------------------------------------------------
Ran 9 tests in 0.440s

OK


## 输出与结论
这里调用完整实验，把三份结果放入notebook_outputs。与脚本示例逐字节比较。

已执行结果表明：质量对数据点求和，密度对区间积分，似然固定数据比较参数；后验还需显式先验。有限网格可能漏掉窄峰，普通乘积或密度可能下溢。模型假设与数值检查不能相互替代。

In [14]:
summary = ex.run_experiment(base/'notebook_outputs')
for name in ('bernoulli.csv','gaussian_integrals.csv','summary.json'):
    assert (base/'outputs'/name).read_bytes() == (base/'notebook_outputs'/name).read_bytes()
print('三份输出与脚本示例逐字节相同。')
print({key:summary[key] for key in ('successes','peak_sigma_0_1','narrow_peak_coarse_area','narrow_peak_fine_area')})

三份输出与脚本示例逐字节相同。
{'successes': 3, 'peak_sigma_0_1': 3.989422804014326, 'narrow_peak_coarse_area': 2.9734390294686117e-05, 'narrow_peak_fine_area': 1.0000000000000009}


## 来源与执行范围

公式、接口和资料核查记录见source_checks.md。阅读顺序和全部手算详解见lecture.md、lab.md和answers.md。作者使用真实ipykernel InProcessKernel在新Python进程中顺序执行所有代码单元；未测浏览器交互、Anaconda新安装和socket内核传输。